# NYC Urban Forestry Analysis — Data Exploration

**Phase 1: Source validation and initial quality audit**

This notebook begins a reproducible analysis of the official NYC Open Data 2015 Street Tree Census.

## Research question

**How do street-tree health patterns differ across NYC boroughs and common species, and where might maintenance attention be prioritized?**

### Phase 1 objectives

1. Load selected variables directly from the official source.
2. Confirm the number of observations and inspect the schema.
3. Measure missing values and duplicate tree identifiers.
4. Record observations before making cleaning decisions.

> This is an observational dataset. Patterns can support further investigation, but they do not by themselves establish cause and effect.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

print("pandas version:", pd.__version__)

## Official dataset

- Dataset: [2015 Street Tree Census – Tree Data](https://data.cityofnewyork.us/Environment/2015-Street-Tree-Census-Tree-Data/uvpi-gqnh)
- Publisher: NYC Department of Parks & Recreation
- NYC Open Data identifier: `uvpi-gqnh`
- Collection period: May 2015–October 2016

Only columns relevant to the research question are loaded. This reduces memory use without changing the number of records.

In [ ]:
DATA_URL = (
    "https://data.cityofnewyork.us/api/views/uvpi-gqnh/"
    "rows.csv?accessType=DOWNLOAD"
)

USE_COLUMNS = [
    "tree_id", "tree_dbh", "status", "health", "spc_common",
    "steward", "guards", "sidewalk", "problems", "postcode",
    "boroname", "nta_name", "latitude", "longitude"
]

trees_raw = pd.read_csv(
    DATA_URL,
    usecols=USE_COLUMNS,
    low_memory=False
)

print(f"Loaded {trees_raw.shape[0]:,} rows and {trees_raw.shape[1]} selected columns.")

## First look

The following cells inspect the data without modifying it.

In [ ]:
display(trees_raw.head())
display(trees_raw.dtypes.rename("dtype").to_frame())

In [ ]:
audit = pd.DataFrame({
    "dtype": trees_raw.dtypes.astype(str),
    "missing_count": trees_raw.isna().sum(),
    "missing_percent": (trees_raw.isna().mean() * 100).round(2),
    "unique_values": trees_raw.nunique(dropna=True)
}).sort_values("missing_percent", ascending=False)

duplicate_tree_ids = trees_raw["tree_id"].duplicated().sum()

display(audit)
print(f"Duplicate tree_id values: {duplicate_tree_ids:,}")
print("\nStatus counts:")
display(trees_raw["status"].value_counts(dropna=False))
print("\nHealth counts:")
display(trees_raw["health"].value_counts(dropna=False))

## Observation checkpoint

After running the cells above, record what the output shows before cleaning anything:

- How many records and selected columns loaded?
- Are tree IDs unique?
- Which fields have the most missing values?
- How are `status` and `health` related?
- Which findings should affect the definition of the analysis population?

**Do not fill this section from expectation. Base every statement on the displayed output.**